incremental data

In [3]:
# ============================================================
# BRONZE LAYER - UPDATED FOR CI/CD (USES GITHUB SECRETS)
# ============================================================

import os
from pyspark.sql import functions as F
from pyspark.sql.types import TimestampType
from datetime import datetime

# ============================================================
# LOAD CONFIGURATION FROM ENVIRONMENT VARIABLES
# ============================================================

# Get environment (DEV, TEST, PROD) - Default to DEV
try:
    env = dbutils.widgets.get("environment")
    print(f"📌 Environment from widget: {env}")
except:
    env = os.getenv('ENV', 'DEV')
    print(f"📌 Environment from env var: {env}")

# Get storage account and access key from environment (GitHub Secrets)
storage_account_name = os.getenv('STORAGE_ACCOUNT', 'capstonestorage01')
access_key = os.getenv('STORAGE_ACCESS_KEY')

# Set container names based on environment
if env == 'DEV':
    raw_container = 'raw-dev'
    bronze_container = 'bronze-dev'
    silver_container = 'silver-dev'
    gold_container = 'gold-dev'
elif env == 'TEST':
    raw_container = 'raw-test'
    bronze_container = 'bronze-test'
    silver_container = 'silver-test'
    gold_container = 'gold-test'
else:  # PROD
    raw_container = 'raw'
    bronze_container = 'bronze'
    silver_container = 'silver'
    gold_container = 'gold'

# Configure Spark with access key (from GitHub Secrets)
if access_key:
    spark.conf.set(
        f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
        access_key
    )
    print("✅ Storage access key configured from GitHub Secrets")
else:
    print("⚠️ WARNING: No access key found! Using Azure AD authentication")

spark.conf.set("spark.databricks.photon.photonRowToColumnar.enabled", "false")

# Dynamic paths based on environment
RAW      = f"abfss://{raw_container}@{storage_account_name}.dfs.core.windows.net"
BRONZE   = f"abfss://{bronze_container}@{storage_account_name}.dfs.core.windows.net"
GOLD     = f"abfss://{gold_container}@{storage_account_name}.dfs.core.windows.net"

print(f"\n{'='*55}")
print(f"🏭 ENVIRONMENT: {env}")
print(f"{'='*55}")
print(f"📁 RAW Container: {raw_container}")
print(f"📁 BRONZE Container: {bronze_container}")
print(f"📁 GOLD Container: {gold_container}")
print(f"📍 RAW Path: {RAW}")
print(f"📍 BRONZE Path: {BRONZE}")
print(f"{'='*55}\n")

# ============================================================
# WATERMARK PARAMETER — comes from ADF or defaults to full load
# ============================================================
dbutils.widgets.text("last_load_date", "2019-12-31")
last_load_date = dbutils.widgets.get("last_load_date")
print(f"📅 Watermark received : {last_load_date}")
print(f"📅 Loading data AFTER : {last_load_date}")

# ============================================================
# AUDIT LOG FUNCTION
# ============================================================
def log_audit(layer, table, status, row_count=0, message=""):
    spark.createDataFrame([{
        "layer"       : layer,
        "table_name"  : table,
        "status"      : status,
        "row_count"   : row_count,
        "message"     : message,
        "environment" : env,
        "processed_at": str(datetime.now())
    }]).write.format("delta").mode("append") \
      .option("mergeSchema", "true") \
      .save(f"{GOLD}/audit_log")
    print(f"  📋 AUDIT | {status.upper():<8} | {table} | rows: {row_count:,}")

# ============================================================
# MAIN
# ============================================================
try:
    ingestion_time = F.lit(datetime.now()).cast(TimestampType())

    # ── MASTER FILES (always full overwrite) ──────────────────
    print("\n📁 Loading Master Files (full overwrite)...")

    df_customer = spark.read \
        .option("header", "true").option("inferSchema", "true") \
        .csv(f"{RAW}/customer_master.csv") \
        .withColumn("ingestion_timestamp", ingestion_time) \
        .withColumn("source_file", F.lit("customer_master.csv"))
    df_customer.write.format("delta").mode("overwrite") \
        .option("overwriteSchema", "true") \
        .save(f"{BRONZE}/customer_master")
    log_audit("bronze", "customer_master", "success", df_customer.count())

    df_account = spark.read \
        .option("header", "true").option("inferSchema", "true") \
        .csv(f"{RAW}/account_master.csv") \
        .withColumn("ingestion_timestamp", ingestion_time) \
        .withColumn("source_file", F.lit("account_master.csv"))
    df_account.write.format("delta").mode("overwrite") \
        .option("overwriteSchema", "true") \
        .save(f"{BRONZE}/account_master")
    log_audit("bronze", "account_master", "success", df_account.count())

    df_branch = spark.read \
        .option("header", "true").option("inferSchema", "true") \
        .csv(f"{RAW}/branch_channel.csv") \
        .withColumn("ingestion_timestamp", ingestion_time) \
        .withColumn("source_file", F.lit("branch_channel.csv"))
    df_branch.write.format("delta").mode("overwrite") \
        .option("overwriteSchema", "true") \
        .save(f"{BRONZE}/branch_channel")
    log_audit("bronze", "branch_channel", "success", df_branch.count())

    df_deliq = spark.read \
        .option("header", "true").option("inferSchema", "true") \
        .csv(f"{RAW}/repayment_delinquency.csv") \
        .withColumn("ingestion_timestamp", ingestion_time) \
        .withColumn("source_file", F.lit("repayment_delinquency.csv"))
    df_deliq.write.format("delta").mode("overwrite") \
        .option("overwriteSchema", "true") \
        .save(f"{BRONZE}/repayment_delinquency")
    log_audit("bronze", "repayment_delinquency", "success", df_deliq.count())

    # ── TRANSACTIONS — INCREMENTAL APPEND ─────────────────────
    print(f"\n📁 Loading Transactions AFTER {last_load_date} (incremental append)...")

    # Read ALL CSVs in transactions folder
    df_all = spark.read \
        .option("header", "true") \
        .option("inferSchema", "true") \
        .csv(f"{RAW}/transactions/*.csv")

    # Rename existing ingestion_timestamp to avoid collision
    if "ingestion_timestamp" in df_all.columns:
        df_all = df_all.withColumnRenamed("ingestion_timestamp", "source_ingestion_timestamp")

    total_available = df_all.count()
    print(f"  📊 Total available rows  : {total_available:,}")

    # ── APPLY WATERMARK FILTER (incremental) ──
    df_filtered = df_all.filter(
        F.col("transaction_date") > last_load_date
    )
    total_filtered = df_filtered.count()
    print(f"  📊 Rows after watermark  : {total_filtered:,}")
    print(f"  📊 Rows skipped          : {total_available - total_filtered:,}")

    # ── LATE ARRIVAL HANDLING ──
    df_late = df_all.filter(
        (F.col("transaction_date") <= last_load_date) &
        (F.col("source_ingestion_timestamp") > last_load_date)
    )
    late_count = df_late.count()
    print(f"  📊 Late arrival records  : {late_count:,}")

    # ── COMBINE FILTERED + LATE ARRIVALS ──
    df_transactions = df_filtered.union(df_late) \
        .withColumn("transaction_date",    F.to_date("transaction_date")) \
        .withColumn("year",                F.year("transaction_date")) \
        .withColumn("month",               F.month("transaction_date")) \
        .withColumn("ingestion_timestamp", ingestion_time)

    # Cache before write
    df_transactions = df_transactions.cache()
    new_rows = df_transactions.count()

    # ── WRITE TO BRONZE (APPEND — incremental) ──
    df_transactions.write \
        .format("delta") \
        .mode("append") \
        .option("mergeSchema", "true") \
        .partitionBy("product_type", "year") \
        .save(f"{BRONZE}/transactions")

    spark.conf.set("spark.databricks.photon.photonRowToColumnar.enabled", "true")

    final_count = spark.read.format("delta").load(f"{BRONZE}/transactions").count()

    log_audit("bronze", "transactions", "success", new_rows,
              f"watermark={last_load_date}, late_arrivals={late_count}, total_in_bronze={final_count}")

    # ── SUMMARY ───────────────────────────────────────────────
    print("\n" + "=" * 55)
    print(f"  BRONZE LAYER SUMMARY - {env}")
    print("=" * 55)
    tables = {
        "customer_master"       : f"{BRONZE}/customer_master",
        "account_master"        : f"{BRONZE}/account_master",
        "branch_channel"        : f"{BRONZE}/branch_channel",
        "repayment_delinquency" : f"{BRONZE}/repayment_delinquency",
        "transactions"          : f"{BRONZE}/transactions",
    }
    for name, path in tables.items():
        count = spark.read.format("delta").load(path).count()
        print(f"  ✅ {name:<30} {count:>10,} rows")
    print(f"\n  📅 Watermark used        : {last_load_date}")
    print(f"  📅 New rows loaded       : {new_rows:,}")
    print(f"  📅 Total rows in Bronze  : {final_count:,}")
    print("=" * 55)
    print(f"  ✅ BRONZE LAYER COMPLETE! ({env})")
    print("=" * 55)

except Exception as e:
    log_audit("bronze", "transactions", "failed", message=str(e))
    raise

ModuleNotFoundError: No module named 'pyspark'